# Task 4 — RAG Evaluation
### TechNova Support · RAG Project


## 1. Imports & Config

In [14]:
import json, re, time, pathlib
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
import pandas as pd
import gradio as gr

load_dotenv(override=True)

MODEL      = "gpt-4.1-nano"
EVAL_FILE  = pathlib.Path("evaluation/questions.json")
OUTPUT     = pathlib.Path("eval_results.json")
TOP_K      = 5

print(f"Model     : {MODEL}")
print(f"Eval file : {EVAL_FILE.resolve()}")
print(f"Output    : {OUTPUT.resolve()}")

Model     : gpt-4.1-nano
Eval file : E:\TechNova_Support_Final\evaluation\questions.json
Output    : E:\TechNova_Support_Final\eval_results.json


## 2. Load Tasks 1, 2 & 3

In [15]:
%run Task1_Ingestion.ipynb
%run Task2_VectorStore.ipynb
%run Task3_RAG_Pipeline.ipynb
print(f"\nPipeline ready — {vectorstore._collection.count()} chunks in vector store")

Imports OK
Loaded 34 documents from knowledge-base/

  [company             ]  company\about.md
  [company             ]  company\careers.md
  [company             ]  company\contact.md
  [company             ]  company\sustainability.md
  [faqs                ]  faqs\account.md
  [faqs                ]  faqs\payments.md
  [faqs                ]  faqs\returns.md
  [faqs                ]  faqs\shipping.md
  [faqs                ]  faqs\software_updates.md
  [faqs                ]  faqs\trade_in.md
  [faqs                ]  faqs\troubleshooting.md
  [faqs                ]  faqs\warranty.md
  [policies            ]  policies\privacy_policy.md
  [policies            ]  policies\return_policy.md
  [policies            ]  policies\shipping_policy.md
  [policies            ]  policies\terms_of_service.md
  [policies            ]  policies\warranty_policy.md
  [products            ]  products\nova_earbuds.md
  [products            ]  products\nova_laptop_pro.md
  [products            ]  produc

e:\TechNova_Support_Final\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)


Imports OK
Loaded 34 documents from knowledge-base/

  [company             ]  company\about.md
  [company             ]  company\careers.md
  [company             ]  company\contact.md
  [company             ]  company\sustainability.md
  [faqs                ]  faqs\account.md
  [faqs                ]  faqs\payments.md
  [faqs                ]  faqs\returns.md
  [faqs                ]  faqs\shipping.md
  [faqs                ]  faqs\software_updates.md
  [faqs                ]  faqs\trade_in.md
  [faqs                ]  faqs\troubleshooting.md
  [faqs                ]  faqs\warranty.md
  [policies            ]  policies\privacy_policy.md
  [policies            ]  policies\return_policy.md
  [policies            ]  policies\shipping_policy.md
  [policies            ]  policies\terms_of_service.md
  [policies            ]  policies\warranty_policy.md
  [products            ]  products\nova_earbuds.md
  [products            ]  products\nova_laptop_pro.md
  [products            ]  produc

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Loaded existing vector store (104 chunks). Skipping rebuild.

Final collection size : 104
Sample metadata       : {'category': 'company', 'start_index': 0, 'source': 'company\\about.md'}
Retrieval sanity checks:

  Q: 'What is the return window for NovaCare+ members?'
  filter=policies
  sources=['policies\\return_policy.md', 'policies\\warranty_policy.md', 'policies\\return_policy.md']

  Q: 'How much is the Nova Phone X1?'
  filter=None
  sources=['products\\nova_phone_x1.md', 'products\\nova_phone_x1.md', 'products\\nova_tablet.md']

  Q: 'cracked watch screen ticket'
  filter=None
  sources=['support_tickets\\ticket_004.md', 'support_tickets\\ticket_004.md', 'support_tickets\\ticket_008.md']

  Q: 'purple banana spaceship'
  filter=None
  sources=['products\\nova_speaker.md', 'products\\nova_phone_x1.md', 'products\\nova_speaker.md']
Model : gpt-4.1-nano  |  Embed : all-MiniLM-L6-v2
Imports OK
Loaded 34 documents from knowledge-base/

  [company             ]  company\about.md
  

e:\TechNova_Support_Final\.venv\Lib\site-packages\nbformat\__init__.py:96: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  validate(nb)
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✓ Loaded existing vector store (104 chunks). Skipping rebuild.

Final collection size : 104
Sample metadata       : {'category': 'company', 'start_index': 0, 'source': 'company\\about.md'}
Retrieval sanity checks:

  Q: 'What is the return window for NovaCare+ members?'
  filter=policies
  sources=['policies\\return_policy.md', 'policies\\warranty_policy.md', 'policies\\return_policy.md']

  Q: 'How much is the Nova Phone X1?'
  filter=None
  sources=['products\\nova_phone_x1.md', 'products\\nova_phone_x1.md', 'products\\nova_tablet.md']

  Q: 'cracked watch screen ticket'
  filter=None
  sources=['support_tickets\\ticket_004.md', 'support_tickets\\ticket_004.md', 'support_tickets\\ticket_008.md']

  Q: 'purple banana spaceship'
  filter=None
  sources=['products\\nova_speaker.md', 'products\\nova_phone_x1.md', 'products\\nova_speaker.md']

Vector store ready — 104 chunks
=== Policy question ===
The return window for NovaCare+ members is extended to 45 days (source: policies\return_pol


Pipeline ready — 104 chunks in vector store


## 3. Load Official Evaluation Questions


In [16]:
if not EVAL_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find '{EVAL_FILE}'. "
        "Make sure you are running from the project root folder that "
        "contains the evaluation/ subfolder."
    )

with open(EVAL_FILE, encoding='utf-8') as f:
    eval_data = json.load(f)

eval_questions = eval_data['questions']
print(f"Loaded {len(eval_questions)} evaluation questions from {EVAL_FILE}")

for q in eval_questions[:3]:
    print(f"  {q['id']:<6} [{q['category']:<20}] {q['question'][:55]}")
print('  ...')

Loaded 29 evaluation questions from evaluation\questions.json
  q01    [products            ] What is the battery capacity of the Nova Phone X1 and h
  q02    [products            ] How much does the base configuration of the Nova Laptop
  q03    [products            ] Which audio codecs do the Nova Earbuds 2 support?
  ...


## 4. Scoring Helpers


In [17]:
def keyword_score(answer: str, must_mention: list) -> float:
    """Fraction of must_mention keywords present in answer (case-insensitive)."""
    if not must_mention:
        return 1.0
    lower = answer.lower()
    found = sum(1 for kw in must_mention if kw.lower() in lower)
    return round(found / len(must_mention), 3)

JUDGE_PROMPT = """You are a strict but fair exam grader.

Question       : {question}
Expected facts : {expected_facts}
Student answer : {answer}

Score the answer from 0 to 5:
  5 = All expected facts correctly stated
  4 = Most facts correct, minor omission
  3 = Some facts correct
  2 = Few facts, mostly incorrect
  1 = Barely relevant
  0 = Completely wrong or hallucinated

Reply ONLY with a JSON object: {{\"score\": <int>, \"reason\": \"<one sentence>\"}}
"""


def llm_judge(question: str, must_mention: list, answer: str, llm) -> dict:
    """Ask GPT to grade the answer 0-5."""
    prompt = JUDGE_PROMPT.format(
        question       = question,
        expected_facts = ", ".join(must_mention),
        answer         = answer[:800]
    )
    try:
        raw  = llm.invoke([HumanMessage(content=prompt)]).content
        raw  = re.sub(r'```json|```', '', raw).strip()
        data = json.loads(raw)
        return {'score': data.get('score', 0) / 5,
                'reason': data.get('reason', '')}
    except Exception as e:
        return {'score': 0.0, 'reason': f'Judge error: {e}'}


def retrieval_hit(docs: list, expected_sources: list) -> bool:
    """True if at least one expected source appears in retrieved docs."""
    if not expected_sources:
        return True
    rs = {d.metadata.get('source', '') for d in docs}
    return any(exp in s for exp in expected_sources for s in rs)


def is_refusal(answer: str) -> bool:
    """True if the answer contains an honest I-don't-know phrase."""
    phrases = ["don't know", "not have", "cannot",
               "don't have that information", "no information"]
    return any(p in answer.lower() for p in phrases)

## 5. Evaluation Loop


In [18]:
judge_llm = ChatOpenAI(model=MODEL, temperature=0)

per_q          = []
kw_scores      = []
llm_scores     = []
ret_hits       = 0
in_scope_count = 0
oos_ok         = None

print(f'Running {len(eval_questions)} questions through the RAG pipeline...\n')

for q in eval_questions:
    print(f"  {q['id']}...", end=' ', flush=True)

    docs = retrieve(q['question'], k=TOP_K)

    answer = answer_question(q['question'], history=[])

    ks  = keyword_score(answer, q.get('must_mention', []))
    hit = retrieval_hit(docs, q.get('expected_sources', []))

    jg  = llm_judge(q['question'], q.get('must_mention', []), answer, judge_llm)

    if q['category'] != 'out_of_scope':
        in_scope_count += 1
        ret_hits       += int(hit)
    else:
        refused = is_refusal(answer)
        oos_ok  = refused
        ks      = 1.0 if refused else 0.0
        jg['score'] = ks
        print(f"REFUSAL={'OK' if refused else 'FAIL'}  ", end='')

    kw_scores.append(ks)
    llm_scores.append(jg['score'])

    per_q.append({
        'id'               : q['id'],
        'category'         : q['category'],
        'question'         : q['question'],
        'predicted_answer' : answer,
        'retrieved_sources': [d.metadata.get('source') for d in docs],
        'retrieval_hit'    : hit,
        'keyword_score'    : ks,
        'passed'           : ks >= 0.5,
        'llm_score'        : jg['score'],
        'llm_reason'       : jg['reason'],
    })

    print(f"kw={ks:.2f}  llm={jg['score']:.2f}")
    time.sleep(0.3)   # gentle rate-limit


ret_precision = round(ret_hits / in_scope_count, 3) if in_scope_count else 0

summary = {
    'total_questions'    : len(eval_questions),
    'passed'             : sum(1 for r in per_q if r['passed']),
    'failed'             : sum(1 for r in per_q if not r['passed']),
    'retrieval_precision': ret_precision,
    'keyword_score_avg'  : round(sum(kw_scores)  / len(kw_scores),  3),
    'llm_score_avg'      : round(sum(llm_scores) / len(llm_scores), 3),
    'refused_correctly'  : bool(oos_ok),
}

with open(OUTPUT, 'w', encoding='utf-8') as f:
    json.dump({'summary': summary, 'per_question': per_q}, f, indent=2)

print('\n' + '='*55)
print('EVALUATION SUMMARY')
print(f"  Total questions     : {summary['total_questions']}")
print(f"  Passed              : {summary['passed']} ({summary['passed']/summary['total_questions']:.0%})")
print(f"  Failed              : {summary['failed']}")
print(f"  Retrieval precision : {summary['retrieval_precision']:.0%}")
print(f"  Keyword score avg   : {summary['keyword_score_avg']:.0%}")
print(f"  LLM judge score avg : {summary['llm_score_avg']:.0%}")
print(f"  Out-of-scope refused: {summary['refused_correctly']}")
print(f"\nResults saved → {OUTPUT}")

Running 29 questions through the RAG pipeline...

  q01... kw=0.00  llm=0.00
  q02... kw=1.00  llm=1.00
  q03... kw=1.00  llm=1.00
  q04... kw=0.00  llm=0.00
  q05... kw=0.67  llm=0.80
  q06... kw=0.50  llm=1.00
  q07... kw=1.00  llm=0.80
  q08... kw=1.00  llm=1.00
  q09... kw=1.00  llm=1.00
  q10... kw=1.00  llm=1.00
  q11... kw=0.50  llm=0.80
  q12... kw=0.67  llm=0.80
  q13... kw=1.00  llm=0.80
  q14... kw=1.00  llm=1.00
  q15... kw=1.00  llm=1.00
  q16... REFUSAL=OK  kw=1.00  llm=1.00
  q17... kw=1.00  llm=1.00
  q18... kw=1.00  llm=1.00
  q19... kw=0.00  llm=0.00
  q20... kw=1.00  llm=0.80
  q21... kw=1.00  llm=1.00
  q22... kw=0.67  llm=0.80
  q23... kw=1.00  llm=1.00
  q24... kw=1.00  llm=1.00
  q25... kw=1.00  llm=0.80
  q26... kw=0.67  llm=0.80
  q27... kw=1.00  llm=1.00
  q28... kw=0.50  llm=0.80
  q29... kw=1.00  llm=1.00

EVALUATION SUMMARY
  Total questions     : 29
  Passed              : 26 (90%)
  Failed              : 3
  Retrieval precision : 0%
  Keyword score avg   

## 6. Results Dashboard

In [19]:
df = pd.DataFrame([
    {
        'ID'               : r['id'],
        'Category'         : r['category'],
        'Question'         : r['question'][:60] + ('...' if len(r['question']) > 60 else ''),
        'Retrieved Sources': ', '.join(r['retrieved_sources'][:2]),
        'Keyword Score'    : f"{r['keyword_score']:.0%}",
        'LLM Score'        : f"{r['llm_score']:.0%}",
        'LLM Reason'       : r['llm_reason'],
        'Status'           : '✅ PASS' if r['passed'] else '❌ FAIL',
    }
    for r in per_q
])

with gr.Blocks(title='TechNova Evaluation', theme=gr.themes.Soft()) as eval_demo:
    gr.Markdown('#RAG Evaluation — TechNova Support')
    gr.Markdown(
        f"**Total**: {summary['total_questions']} &nbsp;|&nbsp; "
        f"**Passed**: {summary['passed']} ({summary['passed']/summary['total_questions']:.0%}) &nbsp;|&nbsp; "
        f"**Failed**: {summary['failed']} &nbsp;|&nbsp; "
        f"**Keyword Avg**: {summary['keyword_score_avg']:.0%} &nbsp;|&nbsp; "
        f"**Retrieval Precision**: {summary['retrieval_precision']:.0%} &nbsp;|&nbsp; "
        f"**LLM Judge Avg**: {summary['llm_score_avg']:.0%}"
    )
    gr.DataFrame(value=df, label='Per-Question Results', interactive=False)

eval_demo.launch()

Running on local URL:  http://127.0.0.1:7868

To create a public link, set `share=True` in `launch()`.
